# design v1

- first simple collaboration recommender
- direct collaboration scoring
- uses aggregated collaboration table


In [33]:
from __future__ import annotations

from ast import literal_eval
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / "data" / "synthetic"
OUTPUT_DIR = ROOT / "data" / "design_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Load


In [34]:
collab_table = pd.read_csv(DATA_DIR / "collaboration_training_table.csv")
collabs = pd.read_csv(DATA_DIR / "collaborations.csv")

collab_table["lastEventAt"] = pd.to_datetime(collab_table["lastEventAt"], utc=True)
collabs["tagsKey"] = collabs["tagsKey"].apply(literal_eval)
collab_table["tagsKey"] = collab_table["tagsKey"].apply(literal_eval)

feature_cols = [
    "collaboration_favorite",
    "collaboration_like",
    "submission_favorite",
    "submission_like",
    "submission_vote",
    "totalEventWeight",
]

collab_table.head()

,userId,projectId,collaborationId,collaboration_favorite,collaboration_like,submission_favorite,submission_like,submission_vote,totalEventWeight,lastEventAt,status,tags,tagsKey,publishedAt
0,user-0001,project-007,collab-007-04,0,0,1,1,0,3.00,2026-04-24 00:22:43+00:00,submission,"['industrial', 'uk-bass']","[industrial, uk-bass]",2026-01-10T03:32:07Z
1,user-0001,project-003,collab-003-01,0,1,1,3,0,5.75,2026-04-03 07:12:35+00:00,submission,"['breakbeat', 'dub', 'jungle']","[breakbeat, dub, jungle]",2026-01-21T00:43:40Z
2,user-0001,project-009,collab-009-02,0,0,1,3,0,5.00,2026-03-28 11:54:51+00:00,voting,['acid'],[acid],2026-01-16T19:47:28Z
3,user-0002,project-007,collab-007-01,0,0,2,2,0,6.00,2026-04-16 05:06:49+00:00,completed,['ambient'],[ambient],2026-02-04T05:59:49Z
4,user-0002,project-012,collab-012-02,0,0,2,2,0,6.00,2026-04-04 14:53:01+00:00,submission,"['breakbeat', 'dub']","[breakbeat, dub]",2026-04-14T19:16:02Z


## Inspect


In [35]:
print("columns:", list(collab_table.columns))
print("shape:", collab_table.shape)
print("\nstatus counts")
print(collab_table["status"].value_counts())
print("\nfeature summary")
print(collab_table[feature_cols].describe().round(2))

columns: ['userId', 'projectId', 'collaborationId', 'collaboration_favorite', 'collaboration_like', 'submission_favorite', 'submission_like', 'submission_vote', 'totalEventWeight', 'lastEventAt', 'status', 'tags', 'tagsKey', 'publishedAt']
shape: (2043, 14)

status counts
status
submission    1045
voting         629
completed      369
Name: count, dtype: int64

feature summary
       collaboration_favorite  collaboration_like  submission_favorite  \
count                 2043.00             2043.00              2043.00   
mean                     0.19                0.39                 1.04   
std                      0.39                0.49                 0.51   
min                      0.00                0.00                 0.00   
25%                      0.00                0.00                 1.00   
50%                      0.00                0.00                 1.00   
75%                      0.00                1.00                 1.00   
max                      1.0

## Active catalog


In [36]:
active_collabs = collabs[collabs["status"].isin(["submission", "voting"])].copy()
active_collabs = active_collabs[["id", "projectId", "name", "status", "tagsKey", "publishedAt"]].rename(columns={"id": "collaborationId"})
active_collabs.head()

,collaborationId,projectId,name,status,tagsKey,publishedAt
0,collab-001-01,project-001,Collaboration 1-1,voting,"[acid, breakbeat, cinematic]",2026-04-28T14:26:30Z
1,collab-001-02,project-001,Collaboration 1-2,voting,"[breakbeat, lofi, organic]",2025-12-17T02:06:25Z
2,collab-002-01,project-002,Collaboration 2-1,submission,"[ambient, drum-and-bass]",2026-03-23T22:33:55Z
3,collab-002-02,project-002,Collaboration 2-2,voting,"[idm, uk-bass]",2026-01-16T13:18:16Z
4,collab-002-03,project-002,Collaboration 2-3,submission,[minimal],2026-03-20T00:06:58Z


## Baseline parts


In [37]:
# popularity per collaboration
popularity = (
    collab_table
    .groupby(["collaborationId", "projectId"], as_index=False)[feature_cols]
    .sum()
)
popularity["popularityScore"] = (
    1.75 * popularity["collaboration_favorite"]
    + 0.75 * popularity["collaboration_like"]
    + 2.0 * popularity["submission_favorite"]
    + 1.0 * popularity["submission_like"]
    + 3.0 * popularity["submission_vote"]
)

max_pop = popularity["popularityScore"].max() or 1
popularity["popularityScoreNorm"] = popularity["popularityScore"] / max_pop
popularity = popularity[["collaborationId", "projectId", "popularityScoreNorm"]]
popularity.head()

,collaborationId,projectId,popularityScoreNorm
0,collab-001-01,project-001,0.625383
1,collab-001-02,project-001,0.860208
2,collab-002-01,project-002,0.344574
3,collab-002-02,project-002,0.347639
4,collab-002-03,project-002,0.298590


In [38]:
# user tag affinity from prior positive interactions
tag_rows = []
for row in collab_table[["userId", "tagsKey", "totalEventWeight"]].itertuples(index=False):
    tags = row.tagsKey or []
    if not tags:
        continue
    per_tag_weight = row.totalEventWeight / len(tags)
    for tag in tags:
        tag_rows.append({"userId": row.userId, "tag": tag, "weight": per_tag_weight})

user_tag_affinity = pd.DataFrame(tag_rows)
user_tag_affinity = user_tag_affinity.groupby(["userId", "tag"], as_index=False)["weight"].sum()
user_tag_affinity.head()

,userId,tag,weight
0,user-0001,acid,5.000000
1,user-0001,breakbeat,1.916667
2,user-0001,dub,1.916667
3,user-0001,industrial,1.500000
4,user-0001,jungle,1.916667


In [39]:
# user project affinity from prior interactions
user_project_affinity = (
    collab_table
    .groupby(["userId", "projectId"], as_index=False)["totalEventWeight"]
    .sum()
    .rename(columns={"totalEventWeight": "projectAffinity"})
)

max_project_affinity = user_project_affinity.groupby("userId")["projectAffinity"].transform("max")
user_project_affinity["projectAffinityNorm"] = user_project_affinity["projectAffinity"] / max_project_affinity
user_project_affinity.head()

,userId,projectId,projectAffinity,projectAffinityNorm
0,user-0001,project-003,5.75,1.000000
1,user-0001,project-007,3.00,0.521739
2,user-0001,project-009,5.00,0.869565
3,user-0002,project-004,3.75,0.384615
4,user-0002,project-007,6.00,0.615385


## Candidate scoring


In [40]:
seen = collab_table[["userId", "collaborationId"]].drop_duplicates().assign(seen=1)
users = pd.DataFrame({"userId": sorted(collab_table["userId"].unique())})

candidates = users.assign(_k=1).merge(active_collabs.assign(_k=1), on="_k").drop(columns="_k")
candidates = candidates.merge(seen, on=["userId", "collaborationId"], how="left")
candidates = candidates[candidates["seen"].isna()].drop(columns="seen")

candidates = candidates.merge(popularity, on=["collaborationId", "projectId"], how="left")
candidates["popularityScoreNorm"] = candidates["popularityScoreNorm"].fillna(0)

candidates = candidates.merge(
    user_project_affinity[["userId", "projectId", "projectAffinityNorm"]],
    on=["userId", "projectId"],
    how="left",
)
candidates["projectAffinityNorm"] = candidates["projectAffinityNorm"].fillna(0)

tag_pref = user_tag_affinity.groupby("userId").apply(
    lambda x: dict(zip(x["tag"], x["weight"]))
).to_dict()

def tag_score(user_id, tags):
    pref = tag_pref.get(user_id, {})
    if not tags:
        return 0.0
    return sum(pref.get(tag, 0.0) for tag in tags) / len(tags)

candidates["tagScoreRaw"] = [tag_score(u, tags) for u, tags in zip(candidates["userId"], candidates["tagsKey"])]
max_tag = candidates.groupby("userId")["tagScoreRaw"].transform("max").replace(0, 1)
candidates["tagScoreNorm"] = candidates["tagScoreRaw"] / max_tag

candidates["v1Score"] = (
    0.50 * candidates["tagScoreNorm"]
    + 0.30 * candidates["projectAffinityNorm"]
    + 0.20 * candidates["popularityScoreNorm"]
)

candidates.head()

,userId,collaborationId,projectId,name,status,tagsKey,publishedAt,popularityScoreNorm,projectAffinityNorm,tagScoreRaw,tagScoreNorm,v1Score
0,user-0001,collab-001-01,project-001,Collaboration 1-1,voting,"[acid, breakbeat, cinematic]",2026-04-28T14:26:30Z,0.625383,0.0,2.305556,0.461111,0.355632
1,user-0001,collab-001-02,project-001,Collaboration 1-2,voting,"[breakbeat, lofi, organic]",2025-12-17T02:06:25Z,0.860208,0.0,0.638889,0.127778,0.235931
2,user-0001,collab-002-01,project-002,Collaboration 2-1,submission,"[ambient, drum-and-bass]",2026-03-23T22:33:55Z,0.344574,0.0,0.000000,0.000000,0.068915
3,user-0001,collab-002-02,project-002,Collaboration 2-2,voting,"[idm, uk-bass]",2026-01-16T13:18:16Z,0.347639,0.0,0.750000,0.150000,0.144528
4,user-0001,collab-002-03,project-002,Collaboration 2-3,submission,[minimal],2026-03-20T00:06:58Z,0.298590,0.0,0.000000,0.000000,0.059718


## Top recommendations


In [41]:
top_recommendations = (
    candidates
    .sort_values(["userId", "v1Score"], ascending=[True, False])
    .groupby("userId")
    .head(10)
    .copy()
)

top_recommendations["rank"] = top_recommendations.groupby("userId").cumcount() + 1
top_recommendations = top_recommendations[[
    "userId",
    "rank",
    "projectId",
    "collaborationId",
    "status",
    "tagsKey",
    "tagScoreNorm",
    "projectAffinityNorm",
    "popularityScoreNorm",
    "v1Score",
]]
top_recommendations.head(20)

,userId,rank,projectId,collaborationId,status,tagsKey,tagScoreNorm,projectAffinityNorm,popularityScoreNorm,v1Score
23,user-0001,1,project-010,collab-010-03,voting,[acid],1.000000,0.000000,0.604537,0.620907
6,user-0001,2,project-003,collab-003-03,submission,"[cinematic, lofi, minimal]",0.000000,1.000000,0.716738,0.443348
19,user-0001,3,project-009,collab-009-03,submission,[lofi],0.000000,0.869565,0.618026,0.384475
30,user-0001,4,project-012,collab-012-04,submission,"[acid, breakbeat, trance]",0.461111,0.000000,0.767014,0.383958
5,user-0001,5,project-003,collab-003-02,submission,[cinematic],0.000000,1.000000,0.378909,0.375782
13,user-0001,6,project-006,collab-006-02,submission,"[acid, breakbeat, trance]",0.461111,0.000000,0.673820,0.365320
0,user-0001,7,project-001,collab-001-01,voting,"[acid, breakbeat, cinematic]",0.461111,0.000000,0.625383,0.355632
21,user-0001,8,project-009,collab-009-05,voting,[idm],0.000000,0.869565,0.456162,0.352102
20,user-0001,9,project-009,collab-009-04,submission,[ambient],0.000000,0.869565,0.229920,0.306854
29,user-0001,10,project-012,collab-012-02,submission,"[breakbeat, dub]",0.383333,0.000000,0.434090,0.278485


## Sanity evaluation

- checks coverage, active-only filtering, tag alignment, and concentration


In [42]:
history_tags = collab_table.groupby("userId")["tagsKey"].apply(lambda rows: set(t for arr in rows for t in arr)).to_dict()
history_projects = collab_table.groupby("userId")["projectId"].apply(lambda s: set(s)).to_dict()

sanity_eval = top_recommendations.copy()
sanity_eval["tagOverlap"] = [len(set(tags) & history_tags.get(uid, set())) for uid, tags in zip(sanity_eval["userId"], sanity_eval["tagsKey"])]
sanity_eval["sameSeenProject"] = [pid in history_projects.get(uid, set()) for uid, pid in zip(sanity_eval["userId"], sanity_eval["projectId"])]

sanity_summary = {
    "rows": int(len(sanity_eval)),
    "users": int(sanity_eval["userId"].nunique()),
    "unique_recommended_collabs": int(sanity_eval["collaborationId"].nunique()),
    "catalog_coverage_pct": round(100 * sanity_eval["collaborationId"].nunique() / len(collabs), 2),
    "active_only": bool(sanity_eval["status"].isin(["submission", "voting"]).all()),
    "top10_tag_overlap_pct": round(100 * (sanity_eval["tagOverlap"] > 0).mean(), 2),
    "top10_same_project_seen_pct": round(100 * sanity_eval["sameSeenProject"].mean(), 2),
    "top1_tag_overlap_pct": round(100 * (sanity_eval[sanity_eval["rank"] == 1]["tagOverlap"] > 0).mean(), 2),
    "top1_same_project_seen_pct": round(100 * sanity_eval[sanity_eval["rank"] == 1]["sameSeenProject"].mean(), 2),
}

sanity_summary


{'rows': 2400,
 'users': 240,
 'unique_recommended_collabs': 38,
 'catalog_coverage_pct': 80.85,
 'active_only': True,
 'top10_tag_overlap_pct': np.float64(94.54),
 'top10_same_project_seen_pct': np.float64(65.5),
 'top1_tag_overlap_pct': np.float64(100.0),
 'top1_same_project_seen_pct': np.float64(74.17)}

In [43]:
top1_concentration = top_recommendations[top_recommendations["rank"] == 1]["collaborationId"].value_counts().head(10)
top1_concentration


collaborationId
collab-009-03    47
collab-011-02    23
collab-013-02    15
collab-013-05    13
collab-008-01    13
collab-010-03    12
collab-009-02    12
collab-012-04    11
collab-006-01     9
collab-007-02     9
Name: count, dtype: int64

## Holdout test

- hold out the most recent active collaboration per user
- train the simple baseline on the older interactions
- check whether the held-out collaboration is recommended back


In [44]:
eligible_history = collab_table[collab_table["status"].isin(["submission", "voting"])].copy()
eligible_history = eligible_history.sort_values(["userId", "lastEventAt"])

holdout = eligible_history.groupby("userId").tail(1).copy()
holdout_users = holdout["userId"]
train_history = eligible_history.merge(holdout[["userId", "collaborationId"]].assign(_holdout=1), on=["userId", "collaborationId"], how="left")
train_history = train_history[train_history["_holdout"].isna()].drop(columns="_holdout")

valid_users = train_history["userId"].value_counts()
valid_users = set(valid_users[valid_users > 0].index) & set(holdout["userId"])
train_history = train_history[train_history["userId"].isin(valid_users)].copy()
holdout = holdout[holdout["userId"].isin(valid_users)].copy()

len(valid_users), train_history.shape, holdout.shape


(239, (1434, 14), (239, 14))

In [45]:
train_popularity = (
    train_history
    .groupby(["collaborationId", "projectId"], as_index=False)[feature_cols]
    .sum()
)
train_popularity["popularityScore"] = (
    1.75 * train_popularity["collaboration_favorite"]
    + 0.75 * train_popularity["collaboration_like"]
    + 2.0 * train_popularity["submission_favorite"]
    + 1.0 * train_popularity["submission_like"]
    + 3.0 * train_popularity["submission_vote"]
)
train_popularity["popularityScoreNorm"] = train_popularity["popularityScore"] / (train_popularity["popularityScore"].max() or 1)
train_popularity = train_popularity[["collaborationId", "projectId", "popularityScoreNorm"]]

tag_rows = []
for row in train_history[["userId", "tagsKey", "totalEventWeight"]].itertuples(index=False):
    tags = row.tagsKey
    if isinstance(tags, str):
        tags = literal_eval(tags)
    tags = tags or []
    if not tags:
        continue
    per_tag_weight = row.totalEventWeight / len(tags)
    for tag in tags:
        tag_rows.append({"userId": row.userId, "tag": tag, "weight": per_tag_weight})

train_user_tag_affinity = pd.DataFrame(tag_rows).groupby(["userId", "tag"], as_index=False)["weight"].sum()

train_user_project_affinity = (
    train_history
    .groupby(["userId", "projectId"], as_index=False)["totalEventWeight"]
    .sum()
    .rename(columns={"totalEventWeight": "projectAffinity"})
)
max_project_affinity = train_user_project_affinity.groupby("userId")["projectAffinity"].transform("max")
train_user_project_affinity["projectAffinityNorm"] = train_user_project_affinity["projectAffinity"] / max_project_affinity


In [46]:
seen_train = train_history[["userId", "collaborationId"]].drop_duplicates().assign(seen=1)
eval_users = pd.DataFrame({"userId": sorted(valid_users)})

eval_candidates = eval_users.assign(_k=1).merge(active_collabs.assign(_k=1), on="_k").drop(columns="_k")
eval_candidates = eval_candidates.merge(seen_train, on=["userId", "collaborationId"], how="left")
eval_candidates = eval_candidates[eval_candidates["seen"].isna()].drop(columns="seen")
eval_candidates = eval_candidates.merge(train_popularity, on=["collaborationId", "projectId"], how="left")
eval_candidates["popularityScoreNorm"] = eval_candidates["popularityScoreNorm"].fillna(0)
eval_candidates = eval_candidates.merge(train_user_project_affinity[["userId", "projectId", "projectAffinityNorm"]], on=["userId", "projectId"], how="left")
eval_candidates["projectAffinityNorm"] = eval_candidates["projectAffinityNorm"].fillna(0)

train_tag_pref = train_user_tag_affinity.groupby("userId").apply(lambda x: dict(zip(x["tag"], x["weight"]))).to_dict()

def eval_tag_score(user_id, tags):
    pref = train_tag_pref.get(user_id, {})
    if isinstance(tags, str):
        tags = literal_eval(tags)
    if not tags:
        return 0.0
    return sum(pref.get(tag, 0.0) for tag in tags) / len(tags)

eval_candidates["tagScoreRaw"] = [eval_tag_score(u, tags) for u, tags in zip(eval_candidates["userId"], eval_candidates["tagsKey"])]
max_tag = eval_candidates.groupby("userId")["tagScoreRaw"].transform("max").replace(0, 1)
eval_candidates["tagScoreNorm"] = eval_candidates["tagScoreRaw"] / max_tag
eval_candidates["v1Score"] = 0.50 * eval_candidates["tagScoreNorm"] + 0.30 * eval_candidates["projectAffinityNorm"] + 0.20 * eval_candidates["popularityScoreNorm"]

eval_topk = eval_candidates.sort_values(["userId", "v1Score"], ascending=[True, False]).groupby("userId").head(10).copy()
eval_topk["rank"] = eval_topk.groupby("userId").cumcount() + 1
eval_topk.head()


,userId,collaborationId,projectId,name,status,tagsKey,publishedAt,popularityScoreNorm,projectAffinityNorm,tagScoreRaw,tagScoreNorm,v1Score,rank
24,user-0001,collab-010-03,project-010,Collaboration 10-3,voting,[acid],2025-11-17T17:49:24Z,0.637692,0.000000,5.000000,1.000000,0.627538,1
6,user-0001,collab-003-03,project-003,Collaboration 3-3,submission,"[cinematic, lofi, minimal]",2025-12-10T14:39:48Z,0.769175,1.000000,0.000000,0.000000,0.453835,2
31,user-0001,collab-012-04,project-012,Collaboration 12-4,submission,"[acid, breakbeat, trance]",2025-12-05T01:46:22Z,0.804967,0.000000,2.305556,0.461111,0.391549,3
20,user-0001,collab-009-03,project-009,Collaboration 9-3,submission,[lofi],2026-04-24T08:15:36Z,0.623083,0.869565,0.000000,0.000000,0.385486,4
5,user-0001,collab-003-02,project-003,Collaboration 3-2,submission,[cinematic],2026-04-21T13:32:20Z,0.371804,1.000000,0.000000,0.000000,0.374361,5


In [47]:
holdout_eval = holdout[["userId", "collaborationId"]].rename(columns={"collaborationId": "heldOutCollaborationId"})
hits = eval_topk.merge(holdout_eval, on="userId", how="inner")
hits["isHit"] = hits["collaborationId"] == hits["heldOutCollaborationId"]

user_hits = hits.groupby("userId").agg(
    hitAt10=("isHit", "max"),
    reciprocalRank=("rank", lambda s: 1 / s[hits.loc[s.index, "isHit"]].iloc[0] if hits.loc[s.index, "isHit"].any() else 0),
).reset_index()

holdout_summary = {
    "users_evaluated": int(len(user_hits)),
    "hit_rate_at_10": round(float(user_hits["hitAt10"].mean()), 4),
    "mrr_at_10": round(float(user_hits["reciprocalRank"].mean()), 4),
}
holdout_summary


{'users_evaluated': 239, 'hit_rate_at_10': 0.3556, 'mrr_at_10': 0.1208}